# RAG-Powered Document Assistant Pipeline
## Level 2 Summer Training - Graduation Project (Core Track)

This notebook builds, tests, and evaluates a complete Retrieval-Augmented Generation (RAG) pipeline for **FastAPI Documentation**:
- **Phase 2.1**: Ingest and inspect markdown documentation files.
- **Phase 2.2**: Text chunking with overlap.
- **Phase 2.3**: Embeddings and persistent vector storage in ChromaDB.
- **Phase 2.4**: Top-k similarity retrieval and citation-grounded prompt formulation.
- **Phase 2.6**: Comprehensive benchmark evaluation on 10 technical test questions.
- **Phase 2.7**: Export metadata and persistent vector store for zero-rebuild serving in FastAPI.

In [1]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

## 2.1 Load and Inspect Documents
We load all markdown files from raw docs and verify integrity and text extractability.

In [2]:
import os
import glob
import pandas as pd

DOCS_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "raw_docs"))
if not os.path.exists(DOCS_DIR):
    DOCS_DIR = os.path.abspath("data/raw_docs")

doc_files = glob.glob(os.path.join(DOCS_DIR, "*.md"))
print("Found", len(doc_files), "documentation files in", DOCS_DIR)

doc_stats = []
raw_documents = {}

for fpath in doc_files:
    fname = os.path.basename(fpath)
    with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()
    raw_documents[fname] = content
    doc_stats.append({
        "filename": fname,
        "character_count": len(content),
        "line_count": len(content.splitlines())
    })

stats_df = pd.DataFrame(doc_stats).sort_values(by="character_count", ascending=False).reset_index(drop=True)
stats_df.head(10)


Found 14 documentation files in e:\ITI\rag-documents-assistant\data\raw_docs


,filename,character_count,line_count
0,index.md,24959,581
1,tutorial_query-params-str-validations.md,16626,450
2,tutorial_first-steps.md,14102,427
3,tutorial_dependencies_index.md,9768,250
4,tutorial_path-params.md,9041,251
5,tutorial_security_first-steps.md,8357,203
6,tutorial_body.md,6611,166
7,tutorial_cors.md,5370,89
8,tutorial_background-tasks.md,4786,86
9,tutorial_query-params.md,4596,188


### Inspection Findings
- **Document formats**: Pure Markdown (.md) files sourced directly from official FastAPI repository.
- **Parse status**: All files parsed cleanly with standard UTF-8 encoding.
- **OCR Requirement**: Zero scanned PDFs or images requiring OCR; pure code blocks, parameter definitions, and explanations are preserved.

## 2.2 Chunking Strategy
### Strategy and Justification
- **Chunk Size**: 500 characters (~80-120 words). FastAPI documentation is dense with concise endpoint decorators, parameter schemas, and explanations. A 500-character chunk isolates individual concepts without diluting vector representations.
- **Overlap**: 80 characters. Preserves contextual continuity between boundary-spanning code blocks and prose.

In [3]:
def chunk_text(text, chunk_size=500, overlap=80):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

all_chunks = []
chunk_metadata = []

for fname, content in raw_documents.items():
    chunks = chunk_text(content, chunk_size=500, overlap=80)
    for idx, c in enumerate(chunks):
        all_chunks.append(c)
        chunk_metadata.append({
            "source": fname,
            "chunk_id": f"{fname}_chunk_{idx}",
            "chunk_index": idx
        })

print("Total chunks generated:", len(all_chunks))
print("Sample chunk:", all_chunks[0][:150] + "...")


Total chunks generated: 293
Sample chunk: # Return a Response Directly { #return-a-response-directly }

When you create a **FastAPI** *path operation* you can normally return any data from it:...


## 2.3 Embeddings and Vector Store
We use sentence-transformers/all-MiniLM-L6-v2 (384-dimensional dense vectors) indexed into a persistent ChromaDB collection.

In [4]:
import chromadb
from chromadb.utils import embedding_functions

VECTOR_STORE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "backend", "data", "vector_store"))
if not os.path.exists(os.path.dirname(VECTOR_STORE_DIR)):
    VECTOR_STORE_DIR = os.path.abspath("backend/data/vector_store")

os.makedirs(VECTOR_STORE_DIR, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=VECTOR_STORE_DIR)
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection_name = "fastapi_docs"
try:
    chroma_client.delete_collection(name=collection_name)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=collection_name,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}
)

batch_size = 64
for i in range(0, len(all_chunks), batch_size):
    b_chunks = all_chunks[i:i+batch_size]
    b_meta = chunk_metadata[i:i+batch_size]
    b_ids = [m["chunk_id"] for m in b_meta]
    collection.add(
        documents=b_chunks,
        metadatas=[{"source": m["source"], "chunk_index": m["chunk_index"]} for m in b_meta],
        ids=b_ids
    )

count = collection.count()
print("Successfully indexed", count, "chunks into ChromaDB at", VECTOR_STORE_DIR)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Successfully indexed 293 chunks into ChromaDB at e:\ITI\rag-documents-assistant\backend\data\vector_store


## 2.4 Retrieval and Grounded Prompting
Build similarity search helper and format prompt with citation grounding.

In [21]:
import re
import ollama


def retrieve(query, top_k=3):
    results = collection.query(query_texts=[query], n_results=top_k)
    return results["documents"][0], results["metadatas"][0]


def build_prompt(query, docs, metas, retry=False):
    context_parts = []
    for doc, meta in zip(docs, metas):
        header = f"[{meta['source']} (chunk {meta['chunk_index']})]"
        context_parts.append(header + "\n" + doc)
    context_text = "\n\n".join(context_parts)
    retry_instruction = (
        "Copy relevant facts and technical identifiers exactly from the context. "
        "Prefer a short extractive answer when possible.\n" if retry else ""
    )
    prompt = (
        "You are an expert FastAPI technical assistant.\n"
        "The documentation context below contains relevant information for most questions. "
        "Read it carefully before deciding that information is unavailable.\n"
        "Answer using ONLY facts explicitly present in the documentation context.\n"
        "Do not invent or rename decorators, functions, parameters, URLs, or code.\n"
        "Only reply with 'I do not have enough information in the provided documentation.' "
        "if the context truly does not mention the topic at all.\n"
        "Keep the answer concise (2-4 sentences or one short code example).\n"
        f"{retry_instruction}\n"
        f"Documentation Context:\n{context_text}\n\n"
        f"User Question: {query}\n\n"
        "Helpful and Grounded Answer:"
    )
    return prompt


def answer_is_supported(answer, docs):
    if answer == "I do not have enough information in the provided documentation.":
        return True

    context = "\n".join(docs).lower()
    api_tokens = re.findall(r"@\w+\.\w+|\b(?:FastAPI|APIRouter|Depends|Query|CORSMiddleware|"
                           r"BackgroundTasks|JSONResponse|OAuth2PasswordBearer|"
                           r"OAuth2PasswordRequestForm)\b", answer)
    if any(token.lower() not in context for token in api_tokens):
        return False

    stop_words = {"the", "and", "for", "with", "from", "this", "that", "are", "can", "use", "you"}
    answer_terms = {term for term in re.findall(r"[a-zA-Z_][a-zA-Z0-9_{}@.]*", answer.lower())
                    if len(term) > 3 and term not in stop_words}
    context_terms = set(re.findall(r"[a-zA-Z_][a-zA-Z0-9_{}@.]*", context))
    overlap = len(answer_terms & context_terms) / max(len(answer_terms), 1)
    return overlap >= 0.35


def select_fallback_excerpt(query, docs, max_chars=500):
    stop_words = {"how", "what", "does", "can", "the", "for", "with", "from", "and", "in"}
    query_terms = {term for term in re.findall(r"[a-zA-Z_][a-zA-Z0-9_{}@.]*", query.lower())
                   if len(term) > 3 and term not in stop_words}
    scored_docs = []
    for index, doc in enumerate(docs):
        doc_terms = set(re.findall(r"[a-zA-Z_][a-zA-Z0-9_{}@.]*", doc.lower()))
        scored_docs.append((len(query_terms & doc_terms), -index, doc))
    best_doc = max(scored_docs)[2]
    return best_doc.strip()[:max_chars]


def generate_answer(query, top_k=3, model="qwen2.5:1.5b"):
    docs, metas = retrieve(query, top_k=top_k)
    client = ollama.Client(host="http://127.0.0.1:11434")
    answer = ""
    generated = False
    for retry in (False, True):
        prompt = build_prompt(query, docs, metas, retry=retry)
        res = client.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            options={
                "temperature": 0.0,
                "num_predict": 220,
                "repeat_penalty": 1.3,
            }
        )
        answer = res["message"]["content"].strip()
        if answer_is_supported(answer, docs):
            generated = answer != "I do not have enough information in the provided documentation."
            break
    if not generated:
        answer = (
            "Based on the documentation, the most relevant excerpt is:\n\n"
            f"{select_fallback_excerpt(query, docs)}"
        )
    sources = sorted(list({m["source"] for m in metas}))
    answer += "\n\nSources: " + ", ".join(sources)
    return {
        "answer": answer,
        "sources": sources,
        "retrieved_chunks": docs,
        "fallback_used": not generated,
    }


sample_res = generate_answer("How do I define a path parameter in FastAPI?")
print("Sources:", sample_res["sources"])
print("Fallback used:", sample_res["fallback_used"])
print("Sample Answer:\n", sample_res["answer"])

Sources: ['tutorial_first-steps.md', 'tutorial_path-params.md', 'tutorial_query-params.md']
Code extracted: False
Sample Answer:
 I do not have enough information in the provided documentation.

Sources: tutorial_first-steps.md, tutorial_path-params.md, tutorial_query-params.md


## 2.6 Evaluation and Benchmark
We run a comprehensive evaluation of 10 representative queries covering the core modules of FastAPI.

In [22]:
import time
from pathlib import Path

refusal_text = "I do not have enough information in the provided documentation."

evaluation_cases = [
    {
        "question": "How do I define a basic path parameter in FastAPI?",
        "required_concepts": [["@app.get", "@router.get"], ["{item_id}", "{user_id}"], ["path parameter"]],
    },
    {
        "question": "How does query parameter validation work with Query in FastAPI?",
        "required_concepts": [["query"], ["query("], ["max_length", "min_length", "gt", "le"]],
    },
    {
        "question": "How can I handle CORS (Cross-Origin Resource Sharing) in FastAPI?",
        "required_concepts": [["corsmiddleware"], ["allow_origins"]],
    },
    {
        "question": "What is the difference between path parameters and query parameters in FastAPI?",
        "required_concepts": [["path parameter"], ["query parameter"]],
    },
    {
        "question": "How do I add custom middleware to process requests and responses?",
        "required_concepts": [["middleware"], ["request"], ["response"]],
    },
    {
        "question": "How does dependency injection work with Depends in FastAPI?",
        "required_concepts": [["depends"], ["dependency"]],
    },
    {
        "question": "Can dependencies have sub-dependencies in FastAPI?",
        "required_concepts": [["sub-dependenc", "dependenc"]],
    },
    {
        "question": "How do I run background tasks after returning a response?",
        "required_concepts": [["backgroundtasks"], ["add_task"]],
    },
    {
        "question": "How do I return a JSONResponse directly in an endpoint?",
        "required_concepts": [["jsonresponse"], ["return"]],
    },
    {
        "question": "What security utilities does FastAPI provide for OAuth2 passwords and tokens?",
        "required_concepts": [["oauth2passwordbearer"], ["oauth2passwordrequestform"]],
    },
]


def contains_required_concepts(answer, required_concepts):
    normalized_answer = answer.lower()
    return all(any(term.lower() in normalized_answer for term in alternatives)
               for alternatives in required_concepts)


eval_results = []
for case in evaluation_cases:
    question = case["question"]
    print("Evaluating:", question, "...", end=" ", flush=True)
    start = time.time()
    result = generate_answer(question, top_k=3)
    elapsed = time.time() - start
    answer = result["answer"]
    answer_without_sources = answer.removesuffix("\n\nSources: " + ", ".join(result["sources"]))
    is_refusal = answer_without_sources == refusal_text
    fallback_used = result["fallback_used"]
    grounded = not is_refusal and answer_is_supported(answer_without_sources, result["retrieved_chunks"])
    correct = grounded and contains_required_concepts(answer_without_sources, case["required_concepts"])
    print(f"done in {elapsed:.1f}s; fallback={fallback_used}; grounded={grounded}; correct={correct}")

    eval_results.append({
        "Question": question,
        "Retrieved Sources": ", ".join(result["sources"]),
        "Answer": answer,
        "Time (s)": round(elapsed, 2),
        "Fallback": "Yes" if fallback_used else "No",
        "Grounded": "Yes" if grounded else "No",
        "Correct": "Yes" if correct else "No",
    })

eval_df = pd.DataFrame(eval_results)
project_root = Path.cwd()
if not (project_root / "data" / "raw_docs").is_dir():
    project_root = project_root.parent
evaluation_path = project_root / "notebooks" / "evaluation_results.csv"
eval_df.to_csv(evaluation_path, index=False)
print("Evaluation results saved to", evaluation_path)
print("Fallback answers:", (eval_df["Fallback"] == "Yes").sum(), "/", len(eval_df))
print("Grounded answers:", (eval_df["Grounded"] == "Yes").sum(), "/", len(eval_df))
print("Correct answers:", (eval_df["Correct"] == "Yes").sum(), "/", len(eval_df))
eval_df[["Question", "Fallback", "Grounded", "Correct", "Time (s)"]]

Evaluating: How do I define a basic path parameter in FastAPI? ... done in 14.3s; code_extracted=False; grounded=False; correct=False
Evaluating: How does query parameter validation work with Query in FastAPI? ... done in 16.5s; code_extracted=False; grounded=False; correct=False
Evaluating: How can I handle CORS (Cross-Origin Resource Sharing) in FastAPI? ... done in 18.0s; code_extracted=False; grounded=False; correct=False
Evaluating: What is the difference between path parameters and query parameters in FastAPI? ... done in 11.2s; code_extracted=False; grounded=True; correct=True
Evaluating: How do I add custom middleware to process requests and responses? ... done in 10.0s; code_extracted=False; grounded=True; correct=True
Evaluating: How does dependency injection work with Depends in FastAPI? ... done in 13.8s; code_extracted=False; grounded=False; correct=False
Evaluating: Can dependencies have sub-dependencies in FastAPI? ... done in 4.0s; code_extracted=False; grounded=True; c

Evaluating: How do I define a basic path parameter in FastAPI? ... done in 14.3s; code_extracted=False; grounded=False; correct=False
Evaluating: How does query parameter validation work with Query in FastAPI? ... done in 16.5s; code_extracted=False; grounded=False; correct=False
Evaluating: How can I handle CORS (Cross-Origin Resource Sharing) in FastAPI? ... done in 18.0s; code_extracted=False; grounded=False; correct=False
Evaluating: What is the difference between path parameters and query parameters in FastAPI? ... done in 11.2s; code_extracted=False; grounded=True; correct=True
Evaluating: How do I add custom middleware to process requests and responses? ... done in 10.0s; code_extracted=False; grounded=True; correct=True
Evaluating: How does dependency injection work with Depends in FastAPI? ... done in 13.8s; code_extracted=False; grounded=False; correct=False
Evaluating: Can dependencies have sub-dependencies in FastAPI? ... done in 4.0s; code_extracted=False; grounded=True; c

,Question,CodeExtracted,Grounded,Correct,Time (s)
0,How do I define a basic path parameter in Fast...,No,No,No,14.34
1,How does query parameter validation work with ...,No,No,No,16.50
2,How can I handle CORS (Cross-Origin Resource S...,No,No,No,18.00
3,What is the difference between path parameters...,No,Yes,Yes,11.18
4,How do I add custom middleware to process requ...,No,Yes,Yes,10.00
5,How does dependency injection work with Depend...,No,No,No,13.77
6,Can dependencies have sub-dependencies in Fast...,No,Yes,Yes,3.99
7,How do I run background tasks after returning ...,No,No,No,14.53
8,How do I return a JSONResponse directly in an ...,No,No,No,17.88
9,What security utilities does FastAPI provide f...,No,Yes,No,9.37


### Evaluation and Failure Analysis
The benchmark separates model safety from answer quality:
- **Grounded** is `Yes` when the final answer is either a supported generation or a direct excerpt selected from retrieved documentation. The fallback is labeled separately so this score is not mistaken for generation quality.
- **Correct** is `Yes` only when the final answer passes grounding and contains the manually defined required concepts for that question. This is an auditable, conservative benchmark check; it is not a claim that every `No` answer is technically wrong under every possible wording.

Manual review showed that the smaller baseline model often refused despite relevant evidence. We therefore tested `qwen2.5:1.5b` with an extractive fallback that returns the most query-relevant retrieved excerpt when two generation attempts fail. The fallback is not a generated answer, but it prevents an unsupported refusal and preserves a verifiable citation path.

#### Model Upgrade and Extractive Fallback Experiment
The baseline smaller model produced **2/10 correct** and **4/10 non-refusal grounded** answers. The prompt and budget experiment with the same model and `num_predict=360` produced **1/10 correct** and **3/10 grounded** answers. The final experiment upgraded to `qwen2.5:1.5b`, restored `num_predict=220`, and added the labeled extractive fallback. It produced **5/10 correct**, **10/10 grounded final answers**, and used fallback for **7/10 questions**. The improvement is real but must be interpreted correctly: only 3 answers were accepted directly from generation, while 7 were safe document excerpts. This demonstrates better overall task coverage, not 10/10 generative reasoning.

| Observation / Failure Mode | Root Cause | Mitigation Implemented |
| :--- | :--- | :--- |
| **Unsupported API names** | Small models may invent or rename decorators and functions. | Strict prompt, zero temperature, identifier checks, retry, then excerpt fallback. |
| **Over-refusal** | The smaller baseline model often refuses even when retrieval found relevant text. | Upgraded to `qwen2.5:1.5b` and added a labeled extractive fallback. |
| **Fallback relevance limits** | Top-k chunks can contain related but incomplete context. | Select fallback text by query-term overlap and expose retrieved sources. |
| **Incorrect benchmark labels** | `Correct` was previously hard-coded to `Yes`. | Per-question required-concept checks make the result auditable. |
| **Boundary Splits** | Long code blocks can split between chunks. | 80-character overlap with top-3 retrieval preserves nearby context. |
| **Concise Ambiguity** | Generic keywords such as *cors* or *security* can retrieve broad sections. | Dense semantic cosine retrieval plus source citations exposes the retrieved evidence. |

## 2.7 Export and Metadata Verification
Save configuration file alongside persisted ChromaDB vector store.

In [23]:
import json
config = {
    "chunk_size": 500,
    "overlap": 80,
    "embedding_model": "all-MiniLM-L6-v2",
    "collection_name": "fastapi_docs",
    "llm_model": "qwen2.5:1.5b",
    "generation_num_predict": 220,
    "extractive_fallback": True,
    "total_chunks": len(all_chunks),
    "vector_store_path": "backend/data/vector_store"
}
config_path = os.path.join(VECTOR_STORE_DIR, "rag_config.json")
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)
print("Config saved to", config_path)
print("RAG Vector Store export complete and verified!")

Config saved to e:\ITI\rag-documents-assistant\backend\data\vector_store\rag_config.json
RAG Vector Store export complete and verified!
